# Pharmasense — Graph WaveNet Training
**Adaptive Spatio-Temporal GNN for communicable disease outbreak prediction**

## Why Graph WaveNet over ST-GNN
ST-GNN (notebook 01) uses a **fixed adjacency matrix** — if two districts share a border, that edge weight is 1, always.  
Graph WaveNet learns an **adaptive adjacency matrix** from data:
```
A_learned = softmax(ReLU(E₁ · E₂ᵀ))   # E₁, E₂ = learnable node embeddings
A_final = A_fixed + A_learned           # combines prior (geography) + learned (transmission)
```
Disease transmission doesn't follow administrative borders. Dengue in Mumbai propagates to Thane
faster than to a geographically adjacent but rural district — the model learns this.

**Temporal:** WaveNet-style dilated causal convolutions (not GRU).  
Receptive field = 2^(layers-1) × kernel_size = 16 weeks per block, 4 blocks = 64 weeks total.
Fully parallelizable — faster to train than RNN-based approaches.

**Ensemble:** At inference, outputs of ST-GNN + Graph WaveNet are averaged.  
Two models with different inductive biases → variance reduction → better accuracy.

## Before running
1. **Runtime → Change runtime type → T4 GPU**
2. **🔑 Secrets → `PHARMASENSE_DB_PASSWORD`**
3. Run ST-GNN notebook first (01) — save `stgnn_latest.pt`
4. **Runtime → Run All**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abhi-dev99/agtechathon/blob/main/ml/notebooks/03_train_graphwavenet.ipynb)

In [ ]:
import subprocess, sys
r = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                   capture_output=True, text=True)
if r.returncode == 0: print(f'GPU: {r.stdout.strip()}')
else: raise RuntimeError('No GPU. Runtime → T4 GPU')

subprocess.run([sys.executable,'-m','pip','install','-q',
    'torch==2.2.0','--index-url','https://download.pytorch.org/whl/cu118'], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q',
    'torch_geometric','torch_scatter','torch_sparse',
    '-f','https://data.pyg.org/whl/torch-2.2.0+cu118.html'], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q',
    'psycopg2-binary','pandas','numpy','scikit-learn','tqdm','matplotlib'], check=True)
print('Ready.')

In [ ]:
# ── Config ─────────────────────────────────────────────────────────────────────
from google.colab import userdata
DB_CONFIG = dict(
    host='db.xwtvujwdybvjbaucshbt.supabase.co', port=5432,
    dbname='postgres', user='postgres',
    password=userdata.get('PHARMASENSE_DB_PASSWORD'), sslmode='require',
)

DISEASES        = ['dengue', 'malaria', 'cholera']  # communicable, spatially spreading
LOOKBACK_WEEKS  = 24
HORIZON_WEEKS   = 2

# Graph WaveNet hyperparameters
# Sized for N=641 nodes on T4 16GB — won't OOM
GWN_RESIDUAL_CH  = 32    # residual channel width
GWN_DILATION_CH  = 32    # dilation channel width  
GWN_SKIP_CH      = 128   # skip connection channels
GWN_END_CH       = 256   # final FC channels
GWN_EMBED_DIM    = 10    # adaptive adjacency embedding dimension
GWN_KERNEL_SIZE  = 2     # WaveNet kernel size
GWN_BLOCKS       = 4     # number of WaveNet blocks
GWN_LAYERS       = 4     # dilated conv layers per block (dilations: 1,2,4,8)
DROPOUT          = 0.2
LEARNING_RATE    = 1e-3
EPOCHS           = 120
EARLY_STOP_PAT   = 20
TRAIN_SPLIT      = 0.75
VAL_SPLIT        = 0.15

DRIVE_OUTPUT = '/content/drive/MyDrive/pharmasense_models/'

import torch
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}  |  Diseases: {DISEASES}')
print(f'Receptive field: {GWN_KERNEL_SIZE ** GWN_LAYERS * GWN_BLOCKS} weeks')

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')
os.makedirs(DRIVE_OUTPUT, exist_ok=True)

In [ ]:
# ── Load data (same preprocessing as ST-GNN notebook) ─────────────────────────
import psycopg2, pandas as pd, numpy as np
from sklearn.preprocessing import StandardScaler

conn = psycopg2.connect(**DB_CONFIG)
districts_df = pd.read_sql("""
    SELECT id, name, state, centroid_lat, centroid_lon, adjacency_ids,
           COALESCE(population, 50000) AS population
    FROM districts ORDER BY id;
""", conn)
dis_filter = "','".join(DISEASES)
epid_df = pd.read_sql(f"""
    SELECT district_id, disease, week_start_date, cases
    FROM epid_weekly WHERE disease IN ('{dis_filter}')
    ORDER BY week_start_date, district_id;
""", conn)
conn.close()

epid_df['week_start_date'] = pd.to_datetime(epid_df['week_start_date'])

# Graph
dist_ids   = list(districts_df['id'])
id_to_idx  = {d:i for i,d in enumerate(dist_ids)}
N          = len(dist_ids)
D_in       = len(DISEASES)

# Fixed adjacency matrix (normalized) — prior for adaptive adjacency
A_fixed = torch.zeros(N, N)
for _, row in districts_df.iterrows():
    i = id_to_idx[row['id']]
    for nbr in (row['adjacency_ids'] or []):
        j = id_to_idx.get(nbr)
        if j is not None:
            A_fixed[i,j] = 1.0
# Row-normalize
deg = A_fixed.sum(1, keepdim=True).clamp(min=1)
A_fixed = (A_fixed / deg).to(DEVICE)
print(f'Fixed adjacency: {N}x{N}, density={(A_fixed > 0).sum().item()/(N*N)*100:.2f}%')

# Feature tensor [T, N, D]
all_weeks   = sorted(epid_df['week_start_date'].unique())
T           = len(all_weeks)
week_to_idx = {w:i for i,w in enumerate(all_weeks)}

X = np.zeros((T, N, D_in), dtype=np.float32)
for row in epid_df.itertuples():
    t = week_to_idx.get(row.week_start_date)
    n = id_to_idx.get(row.district_id)
    d = DISEASES.index(row.disease) if row.disease in DISEASES else None
    if all(v is not None for v in [t,n,d]):
        X[t,n,d] += row.cases

X = np.log1p(X)
scalers = []
for d in range(D_in):
    sc = StandardScaler()
    X[:,:,d] = sc.fit_transform(X[:,:,d].reshape(-1,1)).reshape(T,N)
    scalers.append(sc)

Y = np.roll(X, -HORIZON_WEEKS, axis=0)
Y[-HORIZON_WEEKS:] = 0
print(f'X: {X.shape} | Fill rate: {(X!=0).sum()/X.size*100:.1f}%')

In [ ]:
# ── Graph WaveNet Model ────────────────────────────────────────────────────────
import torch.nn as nn
import torch.nn.functional as F

class GraphConvolution(nn.Module):
    """Single graph convolution step using adjacency matrix."""
    def __init__(self, in_ch, out_ch, support_len=2):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(support_len * in_ch, out_ch))
        self.bias   = nn.Parameter(torch.zeros(out_ch))
        nn.init.xavier_uniform_(self.weight)

    def forward(self, x, supports):
        # x: [N, in_ch]
        h_list = []
        for A in supports:
            h_list.append(A @ x)  # [N, in_ch]
        h = torch.cat(h_list, dim=-1)  # [N, support_len*in_ch]
        return h @ self.weight + self.bias  # [N, out_ch]


class WaveNetBlock(nn.Module):
    """Single dilated causal conv block with graph convolution."""
    def __init__(self, residual_ch, dilation_ch, skip_ch, kernel_size, dilation):
        super().__init__()
        self.filter_conv = nn.Conv1d(residual_ch, dilation_ch, kernel_size,
                                      dilation=dilation, padding=0)
        self.gate_conv   = nn.Conv1d(residual_ch, dilation_ch, kernel_size,
                                      dilation=dilation, padding=0)
        self.graph_conv  = GraphConvolution(dilation_ch, residual_ch, support_len=2)
        self.skip_conv   = nn.Conv1d(residual_ch, skip_ch, 1)
        self.residual_conv = nn.Conv1d(dilation_ch, residual_ch, 1)
        self.bn          = nn.BatchNorm1d(residual_ch)
        self.dilation    = dilation
        self.kernel_size = kernel_size

    def forward(self, x, supports):
        # x: [N, residual_ch, T]
        residual = x
        pad = (self.kernel_size - 1) * self.dilation
        x_pad = F.pad(x, (pad, 0))  # causal padding

        f = torch.tanh(self.filter_conv(x_pad))
        g = torch.sigmoid(self.gate_conv(x_pad))
        h = f * g  # [N, dilation_ch, T]

        # Graph convolution at each timestep
        T = h.shape[-1]
        h_graph = torch.stack(
            [self.graph_conv(h[:, :, t], supports) for t in range(T)], dim=-1
        )  # [N, residual_ch, T]

        skip = self.skip_conv(h_graph)  # [N, skip_ch, T]
        h_res = self.residual_conv(h)   # [N, residual_ch, T]
        out = self.bn(h_res + residual[:, :, -T:])  # residual connection
        return out, skip


class GraphWaveNet(nn.Module):
    """
    Graph WaveNet: Wu et al. (IJCAI 2019)
    'Graph WaveNet for Deep Spatial-Temporal Graph Modeling'

    Key contribution: adaptive adjacency matrix learned end-to-end.
    A_adaptive = softmax(ReLU(E1 @ E2))  where E1,E2 are N×d learnable embeddings.
    Combined with fixed geographic adjacency as a prior.

    Input:  [N, D_in, T_lookback]
    Output: [N, D_out]
    """
    def __init__(self, num_nodes, in_ch, out_ch,
                 residual_ch=32, dilation_ch=32, skip_ch=128, end_ch=256,
                 embed_dim=10, kernel_size=2, blocks=4, layers=4, dropout=0.2):
        super().__init__()
        self.dropout = dropout

        # Adaptive adjacency embeddings — this is the core novelty
        self.nodevec1 = nn.Parameter(torch.randn(num_nodes, embed_dim))
        self.nodevec2 = nn.Parameter(torch.randn(embed_dim, num_nodes))

        self.input_proj = nn.Conv1d(in_ch, residual_ch, 1)

        self.blocks_list = nn.ModuleList()
        for b in range(blocks):
            for l in range(layers):
                dilation = 2 ** l
                self.blocks_list.append(
                    WaveNetBlock(residual_ch, dilation_ch, skip_ch, kernel_size, dilation)
                )

        self.head = nn.Sequential(
            nn.ReLU(),
            nn.Conv1d(skip_ch, end_ch, 1),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Conv1d(end_ch, out_ch, 1),
        )

    def get_supports(self, A_fixed):
        """Compute adaptive adjacency and combine with fixed."""
        A_adapt = F.softmax(F.relu(self.nodevec1 @ self.nodevec2), dim=1)
        return [A_fixed, A_adapt]

    def forward(self, x, A_fixed):
        # x: [N, D_in, T]
        supports = self.get_supports(A_fixed)
        x = self.input_proj(x)      # [N, residual_ch, T]
        skip_sum = None
        for block in self.blocks_list:
            x, skip = block(x, supports)  # skip: [N, skip_ch, T]
            skip_last = skip[:, :, -1:]   # [N, skip_ch, 1] — last timestep
            skip_sum = skip_last if skip_sum is None else skip_sum + skip_last
        out = self.head(skip_sum).squeeze(-1)  # [N, out_ch]
        return out


gwn = GraphWaveNet(
    num_nodes=N, in_ch=D_in, out_ch=D_in,
    residual_ch=GWN_RESIDUAL_CH, dilation_ch=GWN_DILATION_CH,
    skip_ch=GWN_SKIP_CH, end_ch=GWN_END_CH,
    embed_dim=GWN_EMBED_DIM, kernel_size=GWN_KERNEL_SIZE,
    blocks=GWN_BLOCKS, layers=GWN_LAYERS, dropout=DROPOUT
).to(DEVICE)

params = sum(p.numel() for p in gwn.parameters() if p.requires_grad)
print(f'Graph WaveNet | params: {params:,}')
print(f'Adaptive adjacency embeddings: {N} × {GWN_EMBED_DIM} (learnable)')

In [ ]:
# ── Sliding window samples — different format from ST-GNN ─────────────────────
# GWN expects [N, D, T] not [T, N, D]
samples = []
for t in range(T - LOOKBACK_WEEKS - HORIZON_WEEKS):
    x_win  = X[t:t+LOOKBACK_WEEKS]        # [T, N, D]
    x_gwn  = x_win.transpose(1, 2, 0)     # wait — need [N, D, T] for conv1d
    # actually: numpy transpose (1,2,0) -> [N, D, T]
    x_gwn  = np.transpose(x_win, (1, 2, 0))  # [N, D_in, LOOKBACK]
    y_tgt  = Y[t+LOOKBACK_WEEKS-1]            # [N, D]
    samples.append((x_gwn, y_tgt))

n_train = int(len(samples)*0.75)
n_val   = int(len(samples)*0.15)
train_s = samples[:n_train]
val_s   = samples[n_train:n_train+n_val]
test_s  = samples[n_train+n_val:]
print(f'Samples — Train: {len(train_s)}, Val: {len(val_s)}, Test: {len(test_s)}')

from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

optimizer = AdamW(gwn.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)
criterion = nn.HuberLoss(delta=1.0)
A_dev     = A_fixed.to(DEVICE)

train_losses, val_losses = [], []
best_val, best_state, patience = float('inf'), None, 0

for epoch in tqdm(range(EPOCHS), desc='Graph WaveNet'):
    gwn.train()
    np.random.shuffle(train_s)
    ep_loss = 0.0
    for x_np, y_np in train_s:
        x = torch.tensor(x_np, dtype=torch.float32, device=DEVICE)  # [N, D, T]
        y = torch.tensor(y_np, dtype=torch.float32, device=DEVICE)  # [N, D]
        optimizer.zero_grad()
        pred = gwn(x, A_dev)  # [N, D]
        loss = criterion(pred, y)
        loss.backward()
        nn.utils.clip_grad_norm_(gwn.parameters(), 1.0)
        optimizer.step()
        ep_loss += loss.item()
    train_losses.append(ep_loss / len(train_s))

    gwn.eval()
    with torch.no_grad():
        vl = sum(criterion(gwn(torch.tensor(x,device=DEVICE), A_dev),
                           torch.tensor(y,device=DEVICE)).item()
                 for x,y in val_s) / len(val_s)
    val_losses.append(vl)
    scheduler.step()

    if vl < best_val:
        best_val   = vl
        best_state = {k:v.clone() for k,v in gwn.state_dict().items()}
        patience   = 0
    else:
        patience += 1
        if patience >= EARLY_STOP_PAT:
            print(f'Early stop @ epoch {epoch+1} | best_val={best_val:.5f}')
            break

    if (epoch+1) % 20 == 0:
        print(f'E{epoch+1:3d} | train={train_losses[-1]:.5f} | val={vl:.5f} | best={best_val:.5f}')

plt.figure(figsize=(10,4))
plt.plot(train_losses, label='Train'); plt.plot(val_losses, label='Val')
plt.title('Graph WaveNet Loss'); plt.legend()
plt.savefig(DRIVE_OUTPUT+'gwn_loss.png'); plt.show()
print(f'Best val loss: {best_val:.5f}')

In [ ]:
# ── Evaluate + Inspect Learned Adjacency ──────────────────────────────────────
from sklearn.metrics import mean_absolute_error, r2_score

gwn.load_state_dict(best_state)
gwn.eval()

preds, targets = [], []
with torch.no_grad():
    for x_np, y_np in test_s:
        p = gwn(torch.tensor(x_np,device=DEVICE), A_dev).cpu().numpy()
        preds.append(p); targets.append(y_np)

P = np.vstack(preds); T_arr = np.vstack(targets)
print('Test Metrics:')
print(f'  MAE: {mean_absolute_error(T_arr,P):.4f}')
print(f'  R²:  {r2_score(T_arr.ravel(),P.ravel()):.4f}')
for i,d in enumerate(DISEASES):
    print(f'  {d:10s}: MAE={mean_absolute_error(T_arr[:,i],P[:,i]):.4f}')

# Inspect what the adaptive adjacency learned
with torch.no_grad():
    A_adapt = F.softmax(F.relu(gwn.nodevec1 @ gwn.nodevec2), dim=1).cpu().numpy()

# Top-5 strongest learned connections for first 3 districts
print('\nLearned transmission graph (top 5 connections per district):')
for i in range(3):
    top5 = np.argsort(A_adapt[i])[-5:][::-1]
    dist_name = districts_df.iloc[i]['name']
    print(f'  {dist_name}: {[(districts_df.iloc[j]["name"], round(float(A_adapt[i,j]),3)) for j in top5]}')

# Heatmap of adaptive adjacency for first 50 districts
plt.figure(figsize=(10,8))
plt.imshow(A_adapt[:50,:50], cmap='hot', aspect='auto')
plt.colorbar(); plt.title('Learned Adaptive Adjacency (first 50 districts)')
plt.savefig(DRIVE_OUTPUT+'gwn_adaptive_adjacency.png'); plt.show()

In [ ]:
# ── Save ──────────────────────────────────────────────────────────────────────
from datetime import datetime
artifact = {
    'model_state_dict': best_state,
    'model_config': {
        'num_nodes': N, 'in_ch': D_in, 'out_ch': D_in,
        'residual_ch': GWN_RESIDUAL_CH, 'dilation_ch': GWN_DILATION_CH,
        'skip_ch': GWN_SKIP_CH, 'end_ch': GWN_END_CH,
        'embed_dim': GWN_EMBED_DIM, 'kernel_size': GWN_KERNEL_SIZE,
        'blocks': GWN_BLOCKS, 'layers': GWN_LAYERS, 'dropout': DROPOUT,
    },
    'training_config': {'diseases': DISEASES, 'lookback': LOOKBACK_WEEKS, 'horizon': HORIZON_WEEKS},
    'graph': {'n_districts': N, 'id_to_idx': id_to_idx},
    'normalisation': {'scalers': scalers, 'diseases_order': DISEASES},
    'A_fixed': A_fixed.cpu().numpy(),
    'A_adaptive_learned': A_adapt,  # save the learned adjacency for analysis
    'val_loss': best_val,
    'trained_at': datetime.now().isoformat(),
}
save_path = DRIVE_OUTPUT + 'gwn_latest.pt'
torch.save(artifact, save_path)
print(f'Saved: {save_path}')
print('Also save stgnn_latest.pt from notebook 01 to ml/artifacts/')
print('Inference ensemble: (stgnn_pred + gwn_pred) / 2')

from google.colab import files
files.download(save_path)
print('Download started → ml/artifacts/gwn_latest.pt')